In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

## import

In [1]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [2]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention")
cwd = os.getcwd()
print(cwd)

sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention


In [3]:
import scanpy as sc
import spatialdata as sd
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)


In [4]:
import Multi_Modal_Hadmard as MMR

/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [5]:
import importlib
import Multi_Modal_Hadmard.tools
import Multi_Modal_Hadmard.model

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: Use

SpatialData object, with associated Zarr store: /content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 24689, 17051), (3, 12344, 8525), (3, 6172, 4262), (3, 3086, 2131), (3, 1543, 1065)
│     └── 'morphology_focus': DataTree[cyx] (4, 23912, 34154), (4, 11956, 17077), (4, 5978, 8538), (4, 2989, 4269), (4, 1494, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
│     └── 'nucleus_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (63173, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (63173, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (63036, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (63173, 5006)
with coordi

In [ ]:
adata = sdata.tables["table"]
adata

AnnData object with n_obs × n_vars = 63173 × 5006
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

(63173, 5006)

In [ ]:
adata_omiCLIP = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/cells.h5ad")
adata_omiCLIP

AnnData object with n_obs × n_vars = 63173 × 1
    obs: 'cell_id'
    obsm: 'X_custom'

In [ ]:
# 1. Ensure cell IDs are the index (not just a column)
if 'cell_id' in adata.obs.columns:
    adata.obs.set_index('cell_id', inplace=True)
if 'cell_id' in adata_omiCLIP.obs.columns:
    adata_omiCLIP.obs.set_index('cell_id', inplace=True)

# 2. Align the two objects by cell_id (intersection)
common_ids = adata.obs_names.intersection(adata_omiCLIP.obs_names)

# Optional: check how many matched
print(f"Matched {len(common_ids)} cells out of {adata.n_obs}")

# 3. Reorder both to the same order
adata_c = adata[common_ids, :].copy()
adata_omiCLIP_c = adata_omiCLIP[common_ids, :].copy()

# 4. Add the X_custom matrix to adata_main.obsm
adata_c.obsm['Morpho_Embedding'] = adata_omiCLIP_c.obsm['X_custom']

# 5. Done! Verify
print(adata_c.obsm.keys())

Matched 63173 cells out of 63173
KeysView(AxisArrays with keys: spatial, Morpho_Embedding)


In [ ]:
adata_c.write("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

# Dataset

In [6]:
adata = sc.read_h5ad("../../../Data/Breast_Cancer/ann_data.h5ad")

Founsation Models

In [9]:
rng = np.random.default_rng(42)

edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal(M.shape)

In [7]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [16]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [8]:
edata.obs_names = [str(int(cid[63:])-1) for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['UNI'][edata.obs_names.get_indexer(common_cells)]

In [9]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

In [12]:
from sklearn.decomposition import PCA

pca = PCA(n_components=500)
X_reduced = pca.fit_transform(adata.obsm['Morpho_Embedding'])
adata.obsm['p_Morpho_Embedding'] = X_reduced

Preparing Dataset

In [10]:
adata = MMR.prep_adatas(adata, norm=True, log1p=True)
dataset = MMR.make_dataset(adata, sparse_graph=True)

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.


In [11]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)

Expression torch.Size([167780, 313])
Morpho_Embedding torch.Size([167780, 1536])
Neighborhood_Graph torch.Size([2, 1342240])


In [12]:
importlib.reload(MMR.dataset)
importlib.reload(MMR.model)
importlib.reload(MMR)

<module 'Multi_Modal_Hadmard' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention/../../Multi_Modal_Hadmard/__init__.py'>

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

# Train

## Model Type 0

In [21]:
model_type = 0

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [14]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26803
Epoch 101: loss =  0.21856
Epoch 201: loss =  0.18342
Epoch 301: loss =  0.17729
Epoch 401: loss =  0.14491
Epoch 501: loss =  0.13536
Epoch 601: loss =  0.12600
Epoch 701: loss =  0.12164
Epoch 801: loss =  0.12034
Epoch 901: loss =  0.11879
Epoch 1001: loss =  0.11746
Epoch 1101: loss =  0.11705
Epoch 1201: loss =  0.11637
Epoch 1301: loss =  0.11604
Epoch 1401: loss =  0.11590
Epoch 1501: loss =  0.11573
Epoch 1601: loss =  0.11556
Epoch 1701: loss =  0.11547
Epoch 1801: loss =  0.11527
Epoch 1901: loss =  0.11510
Epoch 2001: loss =  0.11501
Epoch 2101: loss =  0.11473
Epoch 2201: loss =  0.11459
Epoch 2301: loss =  0.11434
Epoch 2401: loss =  0.11394
Epoch 2501: loss =  0.11376
Epoch 2601: loss =  0.11339
Epoch 2701: loss =  0.11317
Epoch 2801: loss =  0.11300
Epoch 2901: loss =  0.11284
Epoch 3001: loss =  0.11279
Epoch 3101: loss =  0.11274
Epoch 3201: loss =  0.11258
Epoch 3301: loss =  0.11248
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [22]:
#H_OPTIMUS
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26803
Epoch 101: loss =  0.21827
Epoch 201: loss =  0.18336
Epoch 301: loss =  0.14950
Epoch 401: loss =  0.13734
Epoch 501: loss =  0.13195
Epoch 601: loss =  0.12470
Epoch 701: loss =  0.12143
Epoch 801: loss =  0.11938
Epoch 901: loss =  0.11828
Epoch 1001: loss =  0.11759
Epoch 1101: loss =  0.11696
Epoch 1201: loss =  0.11643
Epoch 1301: loss =  0.11626
Epoch 1401: loss =  0.11567
Epoch 1501: loss =  0.11537
Epoch 1601: loss =  0.11515
Epoch 1701: loss =  0.11503
Epoch 1801: loss =  0.11492
Epoch 1901: loss =  0.11489
Epoch 2001: loss =  0.11471
Epoch 2101: loss =  0.11454
Epoch 2201: loss =  0.11434
Epoch 2301: loss =  0.11417
Epoch 2401: loss =  0.11405
Epoch 2501: loss =  0.11403
Epoch 2601: loss =  0.11378
Epoch 2701: loss =  0.11360
Epoch 2801: loss =  0.11339
Epoch 2901: loss =  0.11331
Epoch 3001: loss =  0.11327
Epoch 3101: loss =  0.11308
Epoch 3201: loss =  0.11313
Epoch 3301: loss =  0.11288
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [14]:
#UNI
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.26803
Epoch 101: loss =  0.21790
Epoch 201: loss =  0.18356
Epoch 301: loss =  0.14647
Epoch 401: loss =  0.13750
Epoch 501: loss =  0.13436
Epoch 601: loss =  0.12759
Epoch 701: loss =  0.12345
Epoch 801: loss =  0.12000
Epoch 901: loss =  0.11813
Epoch 1001: loss =  0.11724
Epoch 1101: loss =  0.12107
Epoch 1201: loss =  0.11778
Epoch 1301: loss =  0.11616
Epoch 1401: loss =  0.11579
Epoch 1501: loss =  0.11531
Epoch 1601: loss =  0.11490
Epoch 1701: loss =  0.11472
Epoch 1801: loss =  0.11449
Epoch 1901: loss =  0.11488
Epoch 2001: loss =  0.11416
Epoch 2101: loss =  0.11392
Epoch 2201: loss =  0.11381
Epoch 2301: loss =  0.11363
Epoch 2401: loss =  0.11342
Epoch 2501: loss =  0.11324
Epoch 2601: loss =  0.11302
Epoch 2701: loss =  0.11305
Epoch 2801: loss =  0.11281
Epoch 2901: loss =  0.11274
Epoch 3001: loss =  0.11265
Epoch 3101: loss =  0.11251
Epoch 3201: loss =  0.11234
Epoch 3301: loss =  0.11225
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [15]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.11170, morpho_loss =  0.18924

## Model Type 1

In [ ]:
model_type = 1

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

model_type:  1
Epoch 1: gene_loss =  0.26791, morpho_loss =  0.23326
Epoch 101: gene_loss =  0.20686, morpho_loss =  0.22027
Epoch 201: gene_loss =  0.14512, morpho_loss =  0.21250
Epoch 301: gene_loss =  0.12934, morpho_loss =  0.20545
Epoch 401: gene_loss =  0.12200, morpho_loss =  0.20225
Epoch 501: gene_loss =  0.11909, morpho_loss =  0.19993
Epoch 601: gene_loss =  0.11767, morpho_loss =  0.19859
Epoch 701: gene_loss =  0.11722, morpho_loss =  0.19746
Epoch 801: gene_loss =  0.11703, morpho_loss =  0.19643
Epoch 901: gene_loss =  0.11650, morpho_loss =  0.19596
Epoch 1001: gene_loss =  0.11654, morpho_loss =  0.19534
Epoch 1101: gene_loss =  0.11626, morpho_loss =  0.19521
Epoch 1201: gene_loss =  0.11624, morpho_loss =  0.19478
Epoch 1301: gene_loss =  0.11611, morpho_loss =  0.19467
Epoch 1401: gene_loss =  0.11603, morpho_loss =  0.19432
Epoch 1501: gene_loss =  0.11593, morpho_loss =  0.19412
Epoch 1601: gene_loss =  0.11582, morpho_loss =  0.19395
Epoch 1701: gene_loss =  0.1

Steamboat(
  (spatial_gather): BilinearAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.10954, morpho_loss =  0.19044

## Model Type 2

In [ ]:
model_type = 2

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

model_type:  2
Epoch 1: gene_loss =  0.26792, morpho_loss =  0.23326
Epoch 101: gene_loss =  0.20701, morpho_loss =  0.22072
Epoch 201: gene_loss =  0.14523, morpho_loss =  0.21308
Epoch 301: gene_loss =  0.12745, morpho_loss =  0.20646
Epoch 401: gene_loss =  0.12189, morpho_loss =  0.20213
Epoch 501: gene_loss =  0.11752, morpho_loss =  0.19965
Epoch 601: gene_loss =  0.11542, morpho_loss =  0.19807
Epoch 701: gene_loss =  0.11457, morpho_loss =  0.19682
Epoch 801: gene_loss =  0.11411, morpho_loss =  0.19580
Epoch 901: gene_loss =  0.11379, morpho_loss =  0.19496
Epoch 1001: gene_loss =  0.11357, morpho_loss =  0.19430
Epoch 1101: gene_loss =  0.11330, morpho_loss =  0.19380
Epoch 1201: gene_loss =  0.11305, morpho_loss =  0.19326
Epoch 1301: gene_loss =  0.11271, morpho_loss =  0.19283
Epoch 1401: gene_loss =  0.11240, morpho_loss =  0.19240
Epoch 1501: gene_loss =  0.11224, morpho_loss =  0.19212
Epoch 1601: gene_loss =  0.11198, morpho_loss =  0.19189
Epoch 1701: gene_loss =  0.1

Steamboat(
  (spatial_gather): BilinearAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.10763, morpho_loss =  0.18602

## Model Type 3

In [ ]:
model_type = 3

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

model_type:  3
Epoch 1: gene_loss =  0.26792, morpho_loss =  0.23327
Epoch 101: gene_loss =  0.20671, morpho_loss =  0.22053
Epoch 201: gene_loss =  0.14006, morpho_loss =  0.21260
Epoch 301: gene_loss =  0.12455, morpho_loss =  0.20608
Epoch 401: gene_loss =  0.12138, morpho_loss =  0.20320
Epoch 501: gene_loss =  0.11948, morpho_loss =  0.20158
Epoch 601: gene_loss =  0.11632, morpho_loss =  0.19991
Epoch 701: gene_loss =  0.11323, morpho_loss =  0.19853
Epoch 801: gene_loss =  0.11248, morpho_loss =  0.19710
Epoch 901: gene_loss =  0.11206, morpho_loss =  0.19599
Epoch 1001: gene_loss =  0.11176, morpho_loss =  0.19574
Epoch 1101: gene_loss =  0.11143, morpho_loss =  0.19488
Epoch 1201: gene_loss =  0.11114, morpho_loss =  0.19431
Epoch 1301: gene_loss =  0.11078, morpho_loss =  0.19407
Epoch 1401: gene_loss =  0.11080, morpho_loss =  0.19370
Epoch 1501: gene_loss =  0.11050, morpho_loss =  0.19349
Epoch 1601: gene_loss =  0.11045, morpho_loss =  0.19333
Epoch 1701: gene_loss =  0.1

Steamboat(
  (spatial_gather): BilinearAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.10801, morpho_loss =  0.19012

## Model Type 4

In [ ]:
model_type = 4

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

model_type:  4
Epoch 1: gene_loss =  0.26790, morpho_loss =  0.23325
Epoch 101: gene_loss =  0.20727, morpho_loss =  0.22042
Epoch 201: gene_loss =  0.14410, morpho_loss =  0.21392
Epoch 301: gene_loss =  0.12901, morpho_loss =  0.20750
Epoch 401: gene_loss =  0.12593, morpho_loss =  0.20266
Epoch 501: gene_loss =  0.12477, morpho_loss =  0.19910
Epoch 601: gene_loss =  0.12310, morpho_loss =  0.19647
Epoch 701: gene_loss =  0.12148, morpho_loss =  0.19492
Epoch 801: gene_loss =  0.12058, morpho_loss =  0.19380
Epoch 901: gene_loss =  0.11945, morpho_loss =  0.19352
Epoch 1001: gene_loss =  0.11906, morpho_loss =  0.19314
Epoch 1101: gene_loss =  0.11874, morpho_loss =  0.19290
Epoch 1201: gene_loss =  0.11848, morpho_loss =  0.19267
Epoch 1301: gene_loss =  0.11805, morpho_loss =  0.19247
Epoch 1401: gene_loss =  0.11774, morpho_loss =  0.19231
Epoch 1501: gene_loss =  0.11758, morpho_loss =  0.19219
Epoch 1601: gene_loss =  0.11745, morpho_loss =  0.19187
Epoch 1701: gene_loss =  0.1

Steamboat(
  (spatial_gather): BilinearAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=500, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.11449, morpho_loss =  0.18615

## Model Type 5

In [12]:
model_type = 5

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [16]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26815
Epoch 101: loss =  0.20655
Epoch 201: loss =  0.14681
Epoch 301: loss =  0.13300
Epoch 401: loss =  0.12997
Epoch 501: loss =  0.12735
Epoch 601: loss =  0.12508
Epoch 701: loss =  0.12405
Epoch 801: loss =  0.12345
Epoch 901: loss =  0.12315
Epoch 1001: loss =  0.12285
Epoch 1101: loss =  0.12181
Epoch 1201: loss =  0.12167
Epoch 1301: loss =  0.12155
Epoch 1401: loss =  0.12150
Epoch 1501: loss =  0.12148
Epoch 1601: loss =  0.12138
Epoch 1701: loss =  0.12135
Epoch 1801: loss =  0.12129
Epoch 1901: loss =  0.12123
Epoch 2001: loss =  0.12112
Epoch 2101: loss =  0.12098
Epoch 2201: loss =  0.12076
Epoch 2301: loss =  0.12053
Epoch 2401: loss =  0.12046
Epoch 2501: loss =  0.12047
Epoch 2601: loss =  0.12041
Epoch 2701: loss =  0.12044
Stopping criterion met. Final loss =  0.12040


Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [13]:
#UNI
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.26815
Epoch 101: loss =  0.20655
Epoch 201: loss =  0.14681
Epoch 301: loss =  0.13300
Epoch 401: loss =  0.12997
Epoch 501: loss =  0.12735
Epoch 601: loss =  0.12508
Epoch 701: loss =  0.12405
Epoch 801: loss =  0.12345
Epoch 901: loss =  0.12315
Epoch 1001: loss =  0.12285
Epoch 1101: loss =  0.12181
Epoch 1201: loss =  0.12167
Epoch 1301: loss =  0.12155
Epoch 1401: loss =  0.12150
Epoch 1501: loss =  0.12148
Epoch 1601: loss =  0.12138
Epoch 1701: loss =  0.12135
Epoch 1801: loss =  0.12129
Epoch 1901: loss =  0.12123
Epoch 2001: loss =  0.12112
Epoch 2101: loss =  0.12098
Epoch 2201: loss =  0.12076
Epoch 2301: loss =  0.12053
Epoch 2401: loss =  0.12046
Epoch 2501: loss =  0.12047
Epoch 2601: loss =  0.12041
Epoch 2701: loss =  0.12044
Stopping criterion met. Final loss =  0.12040


Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=313, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(alph

In [18]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.11173, morpho_loss =  0.19373

Fri 27.03 - 21:00


Maximum iterations reached. Final Loss:  0.10113

# Check